In [1]:
import numpy as np
import xgboost as xgb
import json as _json
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report, precision_recall_curve

DATA_DIR = r"c:\Users\16960\Desktop\期末论文\模型搭建\data\数据集划分\表格"
MODEL_PATH = r"c:\Users\16960\Desktop\期末论文\模型搭建\data\保存模型\xgboost\model.json"
INFO_PATH = r"c:\Users\16960\Desktop\期末论文\模型搭建\data\数据集划分\表格\preprocess_info.json"

with open(INFO_PATH, encoding='utf-8') as _f:
    _info = _json.load(_f)

_feat_names = _info['feature_cols']
_feat_name_map = _info.get('feature_name_map', {})
_display_name = lambda name: f"{_feat_name_map.get(name, name)} [{name}]"

train = np.load(f"{DATA_DIR}/tabular_train.npz")
val = np.load(f"{DATA_DIR}/tabular_val.npz")
test = np.load(f"{DATA_DIR}/tabular_test.npz")

X_train_full, y_train = train['X'], train['y']
X_val_full, y_val = val['X'], val['y']
X_test_full, y_test = test['X'], test['y']

# 去掉训练集零方差特征，避免常量列干扰建模
keep_idx = [i for i in range(X_train_full.shape[1]) if np.std(X_train_full[:, i]) > 0]
drop_idx = [i for i in range(X_train_full.shape[1]) if i not in keep_idx]
active_feature_names = [_feat_names[i] for i in keep_idx]
dropped_feature_names = [_feat_names[i] for i in drop_idx]

X_train = X_train_full[:, keep_idx]
X_val = X_val_full[:, keep_idx]
X_test = X_test_full[:, keep_idx]

pos = float(y_train.sum())
neg = float(len(y_train) - y_train.sum())
scale_pos_weight = neg / max(pos, 1.0)

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=active_feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=active_feature_names)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=active_feature_names)

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'scale_pos_weight': scale_pos_weight,
    'max_delta_step': 1,
    'n_jobs': -1,
    'seed': 42,
}

print(f"训练集样本数: {len(y_train):,}，延误占比: {y_train.mean():.4f}，scale_pos_weight: {scale_pos_weight:.4f}")
if dropped_feature_names:
    print("已剔除零方差特征:")
    for name in dropped_feature_names:
        print(f"  - {_display_name(name)}")

model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100,
)

# 在验证集上搜索最优阈值
val_prob = model.predict(dval)
best_f1, best_t = 0.0, 0.5
for t in np.arange(0.05, 0.96, 0.01):
    p = (val_prob >= t).astype(int)
    f = f1_score(y_val, p, zero_division=0)
    if f > best_f1:
        best_f1, best_t = f, float(t)
print(f"Best threshold: {best_t:.2f} (val F1={best_f1:.4f})")

# 使用最优阈值评估测试集
y_prob = model.predict(dtest)
y_pred = (y_prob >= best_t).astype(int)
baseline_acc = max((y_test == 0).mean(), (y_test == 1).mean())

print(f"\nMajority baseline ACC: {baseline_acc:.4f}")
print(f"ACC: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(f"F1:  {f1_score(y_test, y_pred, zero_division=0):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['正常', '延误'], zero_division=0))

model.save_model(MODEL_PATH)

# 特征重要性分析（gain）
_raw = model.get_score(importance_type='gain')
_imp = {}
for k, v in _raw.items():
    name = k if k in active_feature_names else k
    _imp[name] = _imp.get(name, 0) + v

_imp_sorted = sorted(_imp.items(), key=lambda x: x[1], reverse=True)
print("=== 特征重要性（gain）===")
for name, score in _imp_sorted:
    print(f"  {_display_name(name):40s} {score:>10.1f}")

print(f"\n=== 使用特征 {len(_imp_sorted)}/{len(active_feature_names)} ===")
unused = set(active_feature_names) - set(_imp.keys())
if unused:
    print("本轮未被树使用的特征:")
    for name in sorted(unused):
        print(f"  - {_display_name(name)}")

prec, rec, _ = precision_recall_curve(y_val, val_prob)
max_f1 = max(2 * p * r / (p + r) for p, r in zip(prec[:-1], rec[:-1]) if p + r > 0)
print(f"Max F1 (val): {max_f1:.4f}")


训练集样本数: 4,064,344，延误占比: 0.2168，scale_pos_weight: 3.6122
已剔除零方差特征:
  - 年份 [FL_YEAR]
  - 航班频次 [FLIGHTS]
[0]	train-auc:0.68864	val-auc:0.64610
[100]	train-auc:0.73596	val-auc:0.67026
[168]	train-auc:0.74597	val-auc:0.67059
Best threshold: 0.62 (val F1=0.3473)

Majority baseline ACC: 0.8393
ACC: 0.6525
AUC: 0.6373
F1:  0.3213

              precision    recall  f1-score   support

          正常       0.88      0.68      0.77   1164645
          延误       0.23      0.51      0.32    222930

    accuracy                           0.65   1387575
   macro avg       0.56      0.60      0.54   1387575
weighted avg       0.78      0.65      0.69   1387575

=== 特征重要性（gain）===
  计划出发时间(分钟) [CRS_DEP_TIME_MIN]                1509.5
  承运人 [OP_CARRIER]                              408.9
  出发地降水 [O_PRCP]                                401.2
  出发地风速 [O_WSPD]                                392.6
  出发地气温 [O_TEMP]                                328.0
  月份 [FL_MONTH]                                 274.2
  到达地